In [ ]:
# !pip3 install bibtexparser

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [20]:
import bibtexparser
from bibtexparser.bparser import BibTexParser
from bibtexparser.customization import homogenize_latex_encoding
import re

def normalize_title(title):
    title = homogenize_latex_encoding({"title": title})["title"]
    """Normalize title for comparison: lowercase, remove spaces and punctuation."""
    return re.sub(r'\W+', '', title.lower().strip())

def load_bibtex_file(filepath):
    with open(filepath, 'r', encoding='utf-8') as bibfile:
        parser = BibTexParser(common_strings=True)
        # parser.customization = homogenize_latex_encoding
        return bibtexparser.load(bibfile, parser=parser)

# Load both BibTeX files
detailed = load_bibtex_file("detailed-bibtex.bib")
fallback = load_bibtex_file("google-scholar-export.bib")

# Index entries by normalized title
merged_entries = {}
for entry in fallback.entries:
    title = entry.get("title", "")
    if not title or not entry.get("year"):
        continue
    norm_title = normalize_title(title)
    merged_entries[norm_title] = entry

# Prefer detailed entries if they exist and have a year
for entry in detailed.entries:
    title = entry.get("title", "")
    if not title or not entry.get("year"):
        continue
    norm_title = normalize_title(title)
    merged_entries[norm_title] = entry

# Known journal/booktitle to abbreviation map
known_abbrs = {
    "Journal of Machine Learning Research": "JMLR",
    "IEEE Transactions on Pattern Analysis and Machine Intelligence": "TPAMI",
    "Transactions of the Association for Computational Linguistics": "TACL",
    "Computational Linguistics": "CL",
    "IEEE Computational Intelligence Magazine": "IEEE CIM",
    "IEEE Intelligent Systems": "IEEE IS",
    "IEEE Access": "IEEE Access",
    "ACM Transactions on Information Systems": "TOIS",
    "ACM Transactions on Intelligent Systems and Technology": "TIST",
    "IEEE Transactions on Knowledge and Data Engineering": "TKDE",
    "Pattern Recognition Letters": "PRL",
    "Neurocomputing": "Neurocomputing",
    "Expert Systems with Applications": "ESWA",
    "Annual Conference on Neural Information Processing Systems": "NeurIPS",
    "International Conference on Learning Representations": "ICLR",
    "International Conference on Machine Learning": "ICML",
    "International Joint Conference on Artificial Intelligence": "IJCAI",
    "AAAI Conference on Artificial Intelligence": "AAAI",
    "Conference on Empirical Methods in Natural Language Processing": "EMNLP",
    "Annual Meeting of the Association for Computational Linguistics": "ACL",
    "North American Chapter of the ACL": "NAACL",
    "Conference on Computational Linguistics": "COLING",
    "European Chapter of the ACL": "EACL",
    "Conference on Computer Vision and Pattern Recognition": "CVPR",
    "International Conference on Computer Vision": "ICCV",
    "European Conference on Computer Vision": "ECCV",
    "International Conference on Data Engineering": "ICDE",
    "Very Large Data Bases": "VLDB",
    "ACM SIGMOD International Conference on Management of Data": "SIGMOD",
    "Annual Conference on Web Search and Data Mining": "WSDM",
    "The Web Conference": "WWW",
    "International Conference on Natural Language Processing": "ICON",
    "Language Resources and Evaluation Conference": "LREC",
    "International Conference on Intelligent Text Processing and Computational Linguistics": "CICLing",
}

def infer_abbr(entry):
    if "abbr" not in entry:
        journal = entry.get("journal", "") or entry.get("booktitle", "") or entry.get("publisher", "")
        for known, abbr in known_abbrs.items():
            if known.lower() in journal.lower():
                entry["abbr"] = abbr
                try:
                    if int(entry.get("year", 0)) > 2020:
                        entry["selected"] = "true"
                except ValueError:
                    pass  # Ignore non-integer years
                break

# Apply abbreviation inference to merged entries
for entry in merged_entries.values():
    infer_abbr(entry)
# Create output bib database
output_db = bibtexparser.bibdatabase.BibDatabase()
output_db.entries = list(merged_entries.values())

# Dump to papers.bib
writer = bibtexparser.bwriter.BibTexWriter()
with open("papers-out.bib", "w", encoding="utf-8") as outfile:
    outfile.write(writer.write(output_db))

print(f"✔ Merged {len(output_db.entries)} entries into papers.bib")


✔ Merged 333 entries into papers.bib
